In [2]:
import pandas as pd
import numpy as np
import pickle
print("load successfully")


load successfully


In [8]:
from google.colab import drive
drive.mount('/content/drive')
df=pd.read_csv('/content/drive/MyDrive/WA_Fn-UseC_-Telco-Customer-Churn.csv')
display(df.head())
print("\nData Shape (Rows, Columns):", df.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes



Data Shape (Rows, Columns): (7043, 21)


In [9]:
df=df.drop("customerID", axis=1)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print("Cleaning done! Null values check:")
print(df.isnull().sum().sum())

Cleaning done! Null values check:
0


/tmp/ipykernel_4851/3220405314.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)


In [10]:
from sklearn.preprocessing import LabelEncoder
categorical_cols = df.select_dtypes(include=['object']).columns
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

print("Categorical Encoding complete!")
display(df.head())

Categorical Encoding complete!


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,29.85,0
1,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,1889.50,0
2,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,108.15,1
3,1,0,0,0,45,0,1,0,2,0,2,2,0,0,1,0,0,42.30,1840.75,0
4,0,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,151.65,1


In [11]:
from sklearn.model_selection import train_test_split

# Features aur Target alag karein
X = df.drop('Churn', axis=1)
y = df['Churn']

# Data ko Train aur Test mein divide karein (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data size: {X_train.shape}")
print(f"Testing data size: {X_test.shape}")

Training data size: (5634, 19)
Testing data size: (1409, 19)


In [12]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Model Initialization & Training
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# 3. Prediction & Accuracy
y_pred = model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)

print(f"Model Training Complete! Accuracy: {acc * 100:.2f}%")

Model Training Complete! Accuracy: 79.63%


In [13]:
# Model, Scaler aur Encoders ko ek sath save karein
model_data = {
    'model': model,
    'scaler': scaler,
    'label_encoders': label_encoders
}

with open('churn_model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print("churn_model.pkl file Google Colab session mein successfully save ho gayi hai!")

churn_model.pkl file Google Colab session mein successfully save ho gayi hai!


In [14]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import pickle

# Page Configuration
st.set_page_config(page_title="Customer Churn Predictor", page_icon="📊", layout="wide")

# Saved Model, Scaler, aur Encoders Load Karein
@st.cache_resource
def load_artifacts():
    with open('churn_model.pkl', 'rb') as f:
        artifacts = pickle.load(f)
    return artifacts['model'], artifacts['scaler'], artifacts['label_encoders']

model, scaler, label_encoders = load_artifacts()

# App Header
st.title("📊 Customer Churn Risk Analysis Engine")
st.write("Enter customer service details to calculate churn probability and risk status.")

st.markdown("---")

# Layout: 3 Columns for Form Inputs
col1, col2, col3 = st.columns(3)

with col1:
    st.subheader("Customer Profile")
    gender = st.selectbox("Gender", ["Female", "Male"])
    senior_citizen = st.selectbox("Senior Citizen", ["No", "Yes"])
    partner = st.selectbox("Partner", ["No", "Yes"])
    dependents = st.selectbox("Dependents", ["No", "Yes"])
    tenure = st.slider("Tenure (Months)", min_value=0, max_value=72, value=12)

with col2:
    st.subheader("Subscription Details")
    phone_service = st.selectbox("Phone Service", ["No", "Yes"])
    multiple_lines = st.selectbox("Multiple Lines", ["No", "No phone service", "Yes"])
    internet_service = st.selectbox("Internet Service", ["DSL", "Fiber optic", "No"])
    online_security = st.selectbox("Online Security", ["No", "Yes", "No internet service"])
    online_backup = st.selectbox("Online Backup", ["No", "Yes", "No internet service"])
    device_protection = st.selectbox("Device Protection", ["No", "Yes", "No internet service"])

with col3:
    st.subheader("Billing & Contract")
    tech_support = st.selectbox("Tech Support", ["No", "Yes", "No internet service"])
    streaming_tv = st.selectbox("Streaming TV", ["No", "Yes", "No internet service"])
    streaming_movies = st.selectbox("Streaming Movies", ["No", "Yes", "No internet service"])
    contract = st.selectbox("Contract Type", ["Month-to-month", "One year", "Two year"])
    paperless_billing = st.selectbox("Paperless Billing", ["No", "Yes"])
    payment_method = st.selectbox("Payment Method", [
        "Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"
    ])
    monthly_charges = st.number_input("Monthly Charges ($)", min_value=18.0, max_value=120.0, value=65.0)
    total_charges = st.number_input("Total Charges ($)", min_value=18.0, max_value=9000.0, value= tenure * monthly_charges)

st.markdown("---")

# Prediction Trigger
if st.button("Analyze Churn Risk", type="primary", use_container_width=True):
    # Inputs Dictionary
    raw_inputs = {
        'gender': gender,
        'SeniorCitizen': 1 if senior_citizen == "Yes" else 0,
        'Partner': partner,
        'Dependents': dependents,
        'tenure': tenure,
        'PhoneService': phone_service,
        'MultipleLines': multiple_lines,
        'InternetService': internet_service,
        'OnlineSecurity': online_security,
        'OnlineBackup': online_backup,
        'DeviceProtection': device_protection,
        'TechSupport': tech_support,
        'StreamingTV': streaming_tv,
        'StreamingMovies': streaming_movies,
        'Contract': contract,
        'PaperlessBilling': paperless_billing,
        'PaymentMethod': payment_method,
        'MonthlyCharges': monthly_charges,
        'TotalCharges': total_charges
    }

    # Dataframe Transformation
    input_df = pd.DataFrame([raw_inputs])

    # Categorical Inputs Encoding
    for col, le in label_encoders.items():
        if col in input_df.columns:
            input_df[col] = le.transform(input_df[col])

    # Feature Scaling
    input_scaled = scaler.transform(input_df)

    # Model Inference
    prediction = model.predict(input_scaled)[0]
    probability = model.predict_proba(input_scaled)[0][1] * 100

    # Display Results
    st.subheader("Analysis Results")
    res_col1, res_col2 = st.columns(2)

    with res_col1:
        st.metric(label="Churn Probability", value=f"{probability:.1f}%")

    with res_col2:
        if prediction == 1:
            st.error("⚠️ Status: High Risk (Customer likely to churn)")
            st.warning("Recommended Action: Offer a loyalty discount or long-term contract incentive.")
        else:
            st.success("✅ Status: Low Risk (Customer likely to stay)")
            st.info("Recommended Action: Upsell additional premium services.")

Writing app.py


In [17]:
!pip install pyngrok -q
from pyngrok import ngrok
import os

# Apna Ngrok Authtoken yahan paste karein
ngrok.set_auth_token("3HAoLsdlEUQYjp64eGtJxjOGKCb_38R8kyCcEVjQYBzhQUXwu")

os.system("streamlit run app.py &")
public_url = ngrok.connect(8501)
print(f"Aapki Live Streamlit App ka URL: {public_url}")

Aapki Live Streamlit App ka URL: NgrokTunnel: "https://tattle-donator-kung.ngrok-free.dev" -> "http://localhost:8501"
